# Euler Equation with CentPy in 2d

### Import packages

In [1]:
# Install the centpy package
!pip install centpy

In [2]:
# Import numpy and centpy for the solution
import numpy as np
import centpy

In [3]:
# Imports functions from matplotlib and setup for the animation
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import time

### Equation

We solve the Euler equations in 2D

\begin{equation}
\partial_t
\begin{bmatrix} \rho \\ \rho u_x \\ \rho u_y \\ E \end{bmatrix}
+
\partial_x
\begin{bmatrix} \rho u_x \\ \rho u_x^2 + p \\  \rho u_x u_y \\ (E+p) u_x \end{bmatrix}
+
\partial_y
\begin{bmatrix} \rho u_y \\ \rho u_y u_x \\  \rho u_y^2 +p \\ (E+p) u_y \end{bmatrix}
= 0
\end{equation}

with the equation of state

\begin{equation}
p = (\gamma-1) \left(E-\frac{1}{2} \rho (u_x^2 - u_y^2) \right), \qquad \gamma=1.4
\end{equation}

on the domain $(x,y,t)\in([0,1]\times[0,1]\times[0,0.1])$ with initial data for a *2D Riemann problem*:

\begin{equation}
(\rho, v, p)_{t=0} =
\begin{cases}
(1,0,1) & \text{if} & 0<x\leq0.5 \\
(0.125, 0, 0.1) & \text{if} & 0.5<x<1
\end{cases}
\end{equation}

and Dirichlet boundary data set by initial data on each boundary. The solution is computed using a 200 $\times$ 200 mesh and CFL number 0.75.

In [4]:
pars = centpy.Pars2d(
    x_init=0., x_final=1.,
    y_init=0., y_final=1.,
    J=200, K=200,
    t_final=0.4,
    dt_out=0.005,
    cfl=0.475,
    scheme="sd2",)
pars.gamma = 1.4

In [5]:
# Euler equation
class Euler2d(centpy.Equation2d):

    # Helper functions and definitions for the equation

    def pressure(self, u):
        return (self.gamma - 1.0) * (
            u[:, :, 3] - 0.5 * (u[:, :, 1] ** 2 + u[:, :, 2] ** 2) / u[:, :, 0]
        )

    def euler_data(self):
        gamma = self.gamma

        p_one = 1.5
        p_two = 0.3
        p_three = 0.029
        p_four = 0.3

        upper_right, upper_left, lower_right, lower_left = np.ones((4, 4))

        upper_right[0] = 1.5
        upper_right[1] = 0.0
        upper_right[2] = 0.0
        upper_right[3] = (
            p_one / (gamma - 1.0)
            + 0.5 * (upper_right[1] ** 2 + upper_right[2] ** 2) / upper_right[0]
        )

        upper_left[0] = 0.5323
        upper_left[1] = 1.206 * upper_left[0]
        upper_left[2] = 0.0
        upper_left[3] = ( p_two / (gamma - 1.0)
            + 0.5 * (upper_left[1] ** 2 + upper_left[2] ** 2) / upper_left[0] )

        lower_right[0] = 0.5323
        lower_right[1] = 0.0
        lower_right[2] = 1.206 * lower_right[0]
        lower_right[3] = ( p_four / (gamma - 1.0)
            + 0.5 * (lower_right[1]**2 + lower_right[2]**2) / lower_right[0] )

        lower_left[0] = 0.138
        lower_left[1] = 1.206 * lower_left[0]
        lower_left[2] = 1.206 * lower_left[0]
        lower_left[3] = ( p_three / (gamma - 1.0)
          + 0.5 * (lower_left[1] ** 2 + lower_left[2] ** 2) / lower_left[0] )

        return upper_right, upper_left, lower_right, lower_left

    # Abstract class equation definitions

    def initial_data(self):
        u = np.zeros((self.J + 4, self.K + 4, 4))
        upper_right, upper_left, lower_right, lower_left = self.euler_data()

        # Получаем середину сетки с учетом 2 ghost ячеек с каждой стороны
        mid_J = (self.J // 2) + 2
        mid_K = (self.K // 2) + 2

        # Верхний правый
        u[mid_J:, mid_K:] = upper_right
        # Верхний левый
        u[:mid_J, mid_K:] = upper_left
        # Нижний правый
        u[mid_J:, :mid_K] = lower_right
        # Нижний левый
        u[:mid_J, :mid_K] = lower_left

        return u

    def boundary_conditions(self, u):

        upper_right, upper_left, lower_right, lower_left = self.euler_data()

        if self.odd:
            j = slice(1, -2)
            u[j, 0] = u[j, 1]
            u[j, -2] = u[j, -3]
            u[j, -1] = u[j, -3]

            u[0, j] = u[1, j]
            u[-2, j] = u[-3, j]
            u[-1, j] = u[-3, j]

            # one
            u[-2, -2] = upper_right
            u[-1, -2] = upper_right
            u[-2, -1] = upper_right
            u[-1, -1] = upper_right

            # two
            u[0, -2] = upper_left
            u[0, -1] = upper_left

            # three
            u[0, 0] = lower_left
            u[0, 1] = lower_left
            u[1, 0] = lower_left
            u[1, 1] = lower_left

            # four
            u[-2, 0] = lower_right
            u[-1, 0] = lower_right
            u[-2, 1] = lower_right
            u[-1, 1] = lower_right

        else:

            j = slice(2, -1)
            u[j, 0] = u[j, 2]
            u[j, 1] = u[j, 2]
            u[j, -1] = u[j, -2]

            u[0, j] = u[2, j]
            u[1, j] = u[2, j]
            u[-1, j] = u[-2, j]

            # one
            u[-1, -2] = upper_right
            u[-1, -1] = upper_right

            # two
            u[0, -2] = upper_left
            u[0, -1] = upper_left
            u[1, -2] = upper_left
            u[1, -1] = upper_left

            # three
            u[0, 0] = lower_left
            u[0, 1] = lower_left
            u[1, 0] = lower_left
            u[1, 1] = lower_left

            # four
            u[-1, 0] = lower_right
            u[-1, 1] = lower_right

    def flux_x(self, u):
        f = np.empty_like(u)

        p = self.pressure(u)

        f[:, :, 0] = u[:, :, 1]
        f[:, :, 1] = u[:, :, 1] ** 2 / u[:, :, 0] + p
        f[:, :, 2] = u[:, :, 1] * u[:, :, 2] / u[:, :, 0]
        f[:, :, 3] = (u[:, :, 3] + p) * u[:, :, 1] / u[:, :, 0]

        return f

    def flux_y(self, u):
        g = np.empty_like(u)

        p = self.pressure(u)

        g[:, :, 0] = u[:, :, 2]
        g[:, :, 1] = u[:, :, 1] * u[:, :, 2] / u[:, :, 0]
        g[:, :, 2] = u[:, :, 2] ** 2 / u[:, :, 0] + p
        g[:, :, 3] = (u[:, :, 3] + p) * u[:, :, 2] / u[:, :, 0]

        return g

    def spectral_radius_x(self, u):
        # Если массив 204x204 (имеет теневые ячейки) - берем только центр 200x200.
        # Если массив 200x200 (интерфейсы) - оставляем как есть.
        if u.shape[0] == self.J + 4:
            j0 = slice(2, -2)
            u_core = u[j0, j0]
        else:
            u_core = u

        rho = u_core[..., 0]
        vx = u_core[..., 1] / rho
        vy = u_core[..., 2] / rho
        # Вычисление давления: p = (gamma - 1) * (E - 0.5 * rho * (v_x^2 + v_y^2))
        p = (self.gamma - 1.0) * (u_core[..., 3] - 0.5 * rho * (vx ** 2 + vy ** 2))

        # Защита от отрицательного давления из-за численных погрешностей
        p = np.maximum(p, 1e-10)
        rho = np.maximum(rho, 1e-10)

        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vx) + c

    def spectral_radius_y(self, u):
        if u.shape[0] == self.J + 4:
            j0 = slice(2, -2)
            u_core = u[j0, j0]
        else:
            u_core = u

        rho = u_core[..., 0]
        vx = u_core[..., 1] / rho
        vy = u_core[..., 2] / rho
        p = (self.gamma - 1.0) * (u_core[..., 3] - 0.5 * rho * (vx ** 2 + vy ** 2))

        # Защита от отрицательного давления/плотности
        p = np.maximum(p, 1e-10)
        rho = np.maximum(rho, 1e-10)

        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vy) + c

In [13]:
class EulerSod2d(centpy.Equation2d):

    def _compute_pressure(self, q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        # Защита от деления на ноль (аналогично JAX версии)
        rho_safe = np.maximum(rho, 1e-10)
        u = rhou / rho_safe
        v = rhov / rho_safe
        return (self.gamma - 1.0) * (E - 0.5 * rho * (u**2 + v**2))

    def flux_x(self, q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        rho_safe = np.maximum(rho, 1e-10)
        u = rhou / rho_safe
        p = self._compute_pressure(q)

        f = np.empty_like(q)
        f[..., 0] = rhou
        f[..., 1] = rhou * u + p
        f[..., 2] = rhou * (rhov / rho_safe)
        f[..., 3] = u * (E + p)
        return f

    def flux_y(self, q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        rho_safe = np.maximum(rho, 1e-10)
        v = rhov / rho_safe
        p = self._compute_pressure(q)

        g = np.empty_like(q)
        g[..., 0] = rhov
        g[..., 1] = rhou * v
        g[..., 2] = rhov * v + p
        g[..., 3] = v * (E + p)
        return g

    def spectral_radius_x(self, q):
        rho, rhou = q[..., 0], q[..., 1]
        rho_safe = np.maximum(rho, 1e-10)
        u = rhou / rho_safe
        p = np.maximum(self._compute_pressure(q), 1e-10)
        return np.abs(u) + np.sqrt(self.gamma * p / rho_safe)

    def spectral_radius_y(self, q):
        rho, rhov = q[..., 0], q[..., 2]
        rho_safe = np.maximum(rho, 1e-10)
        v = rhov / rho_safe
        p = np.maximum(self._compute_pressure(q), 1e-10)
        return np.abs(v) + np.sqrt(self.gamma * p / rho_safe)

    def initial_data(self):
        x = self.x

        # 1D задача Сода: разрыв только по оси X
        rho = np.where(x < 0.5, 1.0, 0.125)
        vx = np.zeros_like(x)
        vy = np.zeros_like(x)
        p = np.where(x < 0.5, 1.0, 0.1)

        E = p / (self.gamma - 1.0) + 0.5 * rho * (vx**2 + vy**2)

        u = np.empty((self.J + 4, self.K + 4, 4))
        u[..., 0] = rho
        u[..., 1] = rho * vx
        u[..., 2] = rho * vy
        u[..., 3] = E
        return u

    def boundary_conditions(self, u):
        # Экстраполяция Неймана (нулевой градиент)
        # Копируем крайние внутренние ячейки (индексы 2 и -3) в ghost-зоны
        u[0, :] = u[2, :]
        u[1, :] = u[2, :]
        u[-1, :] = u[-3, :]
        u[-2, :] = u[-3, :]

        u[:, 0] = u[:, 2]
        u[:, 1] = u[:, 2]
        u[:, -1] = u[:, -3]
        u[:, -2] = u[:, -3]
        return u

In [7]:
# Фиксируем маленькую сетку 6x6
pars_cpu = centpy.Pars2d(x_init=0., x_final=1., y_init=0., y_final=1.,
                         J=6, K=6, t_final=0.1, dt_out=0.1, cfl=0.475, scheme="sd2")
pars_cpu.gamma = 1.4
eqn_cpu = Euler2d(pars_cpu)
soln_cpu = centpy.Solver2d(eqn_cpu)

# Берем начальные условия
u_init_cpu = eqn_cpu.initial_data()
soln_cpu.dt = 0.001  # Фиксируем dt вручную для чистоты эксперимента

# В centpy sd2 выполняет один шаг по времени (RK2)
u_step1_cpu = soln_cpu.sd2(u_init_cpu.copy())

# В centpy сетка J+4 на K+4 (с ghost-ячейками), поэтому центр смещен на 2
mid = 6 // 2 + 2

print("--- CPU (centpy) ---")
print("u_init (плотность rho) в центре 2x2:")
print(u_init_cpu[mid-1:mid+1, mid-1:mid+1, 0])
print("\nu_step1 (плотность rho) в центре 2x2:")
print(u_step1_cpu[mid-1:mid+1, mid-1:mid+1, 0])

--- CPU (centpy) ---
u_init (плотность rho) в центре 2x2:
[[0.138  0.5323]
 [0.5323 1.5   ]]

u_step1 (плотность rho) в центре 2x2:
[[0.14314    0.5386259 ]
 [0.5386259  1.49187239]]


### Solution

In [14]:
eqn = EulerSod2d(pars)
t0 = time.time()
soln = centpy.Solver2d(eqn)
soln.solve()
t1 = time.time()
print(f"\n[СPU centpy] Чистое время выполнения: {t1 - t0:.4f} секунд")


[СPU centpy] Чистое время выполнения: 50.2453 секунд


### Animation

In [15]:
# Animation
fig, ax = plt.subplots()
ax.set_xlim(soln.x_init, soln.x_final)
ax.set_ylim(soln.y_init, soln.y_final)

# Извлекаем сетку координат и начальные данные для первого кадра
x_grid = soln.x[1:-1]
y_grid = soln.y[1:-1]
data_init = soln.u_n[0, 1:-1, 1:-1, 0]

# 1. Создаем фоновую тепловую карту с помощью imshow
# Параметр extent привязывает матрицу к реальным координатам
im = ax.imshow(
    data_init,
    extent=[soln.x_init, soln.x_final, soln.y_init, soln.y_final],
    origin='lower',             # Гарантирует, что ось Y направлена вверх
    cmap='coolwarm',             # Или 'magma', 'viridis', 'turbo'
    interpolation='bicubic',    # Сглаживание для профессионального вида
    aspect='auto'               # Позволяет осям масштабироваться независимо
)

# Добавляем цветовую шкалу для наглядности (опционально, но рекомендуется)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Значение u_n')

# 2. Отрисовываем начальные контуры поверх тепловой карты
ax.contour(
    x_grid, y_grid, data_init,
    levels=20,
    colors='black',
    alpha=0.5,
    linewidths=0.5
)

# Функция обновления для каждого кадра анимации
def animate(i):
    # Получаем данные текущего шага
    data = soln.u_n[i, 1:-1, 1:-1, 0]

    # Быстрое обновление тепловой карты (работает быстрее, чем перерисовка)
    im.set_data(data)

    # Если глобальный минимум и максимум меняются со временем,
    # можно раскомментировать следующую строку для динамической шкалы:
    # im.set_clim(vmin=data.min(), vmax=data.max())

    # Удаляем старые линии контуров из коллекции осей
    for c in ax.collections:
        c.remove()

    # Рисуем новые контурные линии для текущего кадра
    ax.contour(
        x_grid, y_grid, data,
        levels=20,
        colors='black',
        alpha=0.5,
        linewidths=0.5
    )

    return [im]

plt.close() # Закрываем статичную фигуру, чтобы она не дублировалась в выводе

# Создаем анимацию
anim = animation.FuncAnimation(fig, animate, frames=soln.Nt, interval=100, blit=False)

# Выводим как HTML5 видео
HTML(anim.to_html5_video())